LIBRARY IMPORTS & DATA LOADING

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("MODULE 3: KPI FEATURE ENGINEERING")
print("=" * 70)

# Load complete military dataset
print("\nLoading military_complete.csv...")
df = pd.read_csv('military_complete.csv')

print(f"Dataset shape: {df.shape}")
print(f"Countries: {len(df)}")

 SECTION 1: CORE KPI CALCULATIONS

In [ ]:
print("\n" + "=" * 70)
print("CALCULATING CORE KPIs...")
print("=" * 70)

# 1. ASSETS PER CAPITA
print("\n1️⃣  Assets per Capita")
df['assets_per_capita'] = (
    (df.get('total_aircraft', 0) + df.get('tanks', 0) + df.get('naval_assets', 0))
    / df['population'].replace(0, np.nan)
)

# 2. DEFENSE BUDGET TO GDP RATIO
print("\n2️⃣  Defense Budget to GDP Ratio")
df['budget_to_gdp_ratio'] = (
    df.get('defense_budget_usd', 0) / df.get('gdp_usd', np.nan).replace(0, np.nan)
)

# 3. PERSONNEL DENSITY
print("\n3️⃣  Personnel Density")
df['personnel_density'] = (
    df.get('total_personnel', 0) / df['population'].replace(0, np.nan)
)

# 4. DEFENSE BUDGET PER SOLDIER
print("\n4️⃣  Defense Budget per Soldier")
df['budget_per_soldier'] = (
    df.get('defense_budget_usd', 0) / df.get('active_personnel', np.nan).replace(0, np.nan)
)

# 5. POWER INDEX RANK GAP
print("\n5️⃣  Power Index Rank Gap")
reference_rank = df.get('power_index_rank', pd.Series([1])).min()
df['power_index_rank_gap'] = (
    pd.to_numeric(df.get('power_index_rank', 1), errors='coerce') - reference_rank
)

# 6. AIR POWER RATIO
print("\n6️⃣  Air Power Ratio")
df['air_power_ratio'] = (
    df.get('total_aircraft', 0) / df.get('active_personnel', np.nan).replace(0, np.nan)
)

# 7. ARMOR INTENSITY INDEX
print("\n7️⃣  Armor Intensity Index")
df['armor_intensity_index'] = (
    df.get('tanks', 0) / df.get('land_area_sq_km', np.nan).replace(0, np.nan)
)

# 8. NAVAL STRENGTH PER COASTLINE
print("\n8️⃣  Naval Strength per Coastline")
df['naval_strength_per_coastline'] = (
    df.get('naval_assets', 0) / df.get('coastline_km', np.nan).replace(0, np.nan)
)

# 9. MILITARY BURDEN INDEX
print("\n9️⃣  Military Burden Index")
df['military_burden_index'] = (
    df.get('defense_budget_usd', 0) / df['population'].replace(0, np.nan)
)

# 10. TOTAL ASSETS
print("\n🔟 Total Assets")
df['total_assets'] = (
    df.get('total_aircraft', 0) + df.get('tanks', 0) + df.get('naval_assets', 0)
)

SECTION 2: COMPOSITE KPIs

In [ ]:
print("\n" + "=" * 70)
print("CALCULATING COMPOSITE KPIs...")
print("=" * 70)

# 11. MILITARY CAPABILITY SCORE (Weighted Composite)
print("\n1️⃣1️⃣  Military Capability Score")
numeric_cols = ['active_personnel', 'total_aircraft', 'tanks', 'naval_assets', 'defense_budget_usd']
normalized = pd.DataFrame()
for col in numeric_cols:
    col_clean = pd.to_numeric(df.get(col, 0), errors='coerce').replace(0, np.nan)
    min_val = col_clean.min()
    max_val = col_clean.max()
    if pd.notna(max_val) and max_val > min_val:
        normalized[col] = ((col_clean - min_val) / (max_val - min_val) * 100).fillna(0)
    else:
        normalized[col] = 0

df['military_capability_score'] = (
    normalized.get('active_personnel', 0) * 0.25 +
    normalized.get('total_aircraft', 0) * 0.20 +
    normalized.get('tanks', 0) * 0.20 +
    normalized.get('naval_assets', 0) * 0.20 +
    normalized.get('defense_budget_usd', 0) * 0.15
).round(2)

# 12. REGIONAL POWER INDEX
print("\n1️⃣2️⃣  Regional Power Index")
if 'region' in df.columns:
    df['regional_power_index'] = (
        df.groupby('region')['total_assets']
        .rank(method='min', ascending=False)
        .astype('Int64')
    )
else:
    df['regional_power_index'] = np.nan

# 13. CONTINENTAL POWER INDEX
print("\n1️⃣3️⃣  Continental Power Index")
if 'continent' in df.columns:
    df['continental_power_index'] = (
        df.groupby('continent')['total_assets']
        .rank(method='min', ascending=False)
        .astype('Int64')
    )
else:
    df['continental_power_index'] = np.nan

SECTION 3: DATA QUALITY / Clean & Standardize

In [ ]:
print("\n" + "=" * 70)
print("QUALITY ASSURANCE...")
print("=" * 70)

# Replace inf with NaN
df = df.replace([np.inf, -np.inf], np.nan)

# Round float columns
float_cols = df.select_dtypes(include=['float64']).columns
for col in float_cols:
    df[col] = df[col].round(4)

SECTION 4: COALITION STRENGTH

In [ ]:
print("\n" + "=" * 70)
print("COALITION STRENGTH ANALYSIS...")
print("=" * 70)

if 'alliance' in df.columns:
    alliance_stats = df.groupby('alliance').agg({
        'total_assets': 'sum',
        'active_personnel': 'sum',
        'defense_budget_usd': 'sum',
        'country': 'count'
    }).rename(columns={'country': 'num_countries'})

    print("\nAlliance Totals:")
    for alliance in alliance_stats.index:
        stats = alliance_stats.loc[alliance]
        print(f"  {alliance}: {int(stats['total_assets']):,} total assets, {int(stats['num_countries'])} countries")

    df = df.merge(
        alliance_stats[['total_assets']].rename(columns={'total_assets': 'alliance_total_assets'}),
        left_on='alliance',
        right_index=True,
        how='left'
    )
else:
    df['alliance_total_assets'] = np.nan

SECTION 5: EXPORTING FINAL XLSX FILE

In [ ]:
print("=" * 70)
print("PREPARING FINAL DATASET...")
print("=" * 70)

# ALL EXISTING COLUMNS + ALL NEW KPIs
kpi_cols = [
    'assets_per_capita', 'budget_to_gdp_ratio', 'personnel_density', 'budget_per_soldier',
    'power_index_rank_gap', 'air_power_ratio', 'armor_intensity_index',
    'naval_strength_per_coastline', 'military_burden_index', 'total_assets',
    'military_capability_score', 'regional_power_index', 'continental_power_index',
    'alliance_total_assets'
]

# Get ALL columns + add KPIs (no duplicates)
all_existing_cols = [col for col in df.columns if col not in kpi_cols]
final_cols = all_existing_cols + [col for col in kpi_cols if col in df.columns]

df_final = df[final_cols].copy()
print(f"Final dataset: {df_final.shape} | {len(final_cols)} columns")

print("=" * 70)
print("EXPORTING FINAL EXCEL FILE...")
print("=" * 70)

# ONE SHEET WITH EVERYTHING
output_file = 'military_final.xlsx'
df_final.to_excel(output_file, sheet_name='Complete_Data_KPIs', index=False)

print(f"✅ CREATED: {output_file}")
